# Exploratory Data Analysis

## 0. imports, data checks

In [0]:
%pip install kaggle
%pip install opencv-python-headless==4.10.0.84

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

CSV_PATH = "/Volumes/workspace/default/ucf_crime/processed/spatiotemporal_ground_truth.csv"
boxes = pd.read_csv(CSV_PATH)

print(boxes.head())
print(boxes.shape)
print(boxes.columns.tolist())
print(boxes.isna().sum())

In [0]:
import cv2

VIDEO_DIR = "/Volumes/workspace/default/ucf_crime/videos"

# videos are stored under nested Kaggle folders: Part-X/Part-X/<Class>/<video_id>.mp4
video_paths = {}
for root, _, files in os.walk(VIDEO_DIR):
    for f in files:
        if f.endswith(".mp4"):
            video_paths[os.path.splitext(f)[0]] = os.path.join(root, f)

vids = boxes["video_id"].unique()
print(f"{len(video_paths)} videos on disk, {len(set(vids) - set(video_paths))} of {len(vids)} ground-truth videos missing")

rows = []
for vid in vids:
    if vid not in video_paths:
        continue
    cap = cv2.VideoCapture(video_paths[vid])
    rows.append({"video_id": vid,
                 "fps": cap.get(cv2.CAP_PROP_FPS),
                 "n_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT))})
    cap.release()

video_metadata = pd.DataFrame(rows)
print(video_metadata["fps"].value_counts())

In [0]:
gt_intervals = (
    boxes.groupby(['video_id', 'interval_id'])  
    .agg(class_=('class', 'first'),
         start_frame=('start_frame', 'first'),
         end_frame=('end_frame', 'first'),
         n_boxes=('frame_number', 'count'))
    .reset_index()
    .rename(columns={'class_': 'class'})
)

print(gt_intervals.shape)

## 1. dataset composition
- _metrics_: videos, intervals and boxes per class, counts, shares, # videos with two intervals
- _plot_ intervals per class
---
- _decision_ -> which classes are too small to report itself, merge or report with warning?

In [0]:
comp = gt_intervals.groupby('class').agg(
    videos = ('video_id', 'nunique'),
    intervals = ('interval_id', 'size'),
    boxes = ('n_boxes', 'sum')
)
comp["interval_share"] = (comp["intervals"] / comp["intervals"].sum()).round(3)
comp = comp.sort_values("intervals", ascending=False)
display(comp)

### videos with two intervals

In [0]:
per_video = gt_intervals.groupby("video_id").size()
two = per_video[per_video == 2]
print(len(two), "videos with 2 intervals:", list(two.index))

### plot: intervals per class

In [0]:
comp["intervals"].sort_values().plot.barh(figsize=(6, 5), title="Intervals per class")
plt.xlabel("# intervals"); plt.tight_layout(); plt.show()

### decision cell 
classes too small to report alone

In [0]:
MIN_N = 10
comp["small"] = comp["intervals"] < MIN_N
display(comp[comp["small"]])

abuse (2 videos), assault (3 videos) flagged.

## 2. temporal properties
- _metrics_ (interval duration via median, IQR, minmax) -> how long each event lasts ((end_f-start_f) / fps)
- _window length_ (interval + padding) -> clip given to model
- _model frames_ (min(64, window * 2)) -> snapshots per window, 2 = fps rate, 64 = frame cap
- _effective sampling_ (frames/window) -> snapshots per sec model sees
    - same 64 frames: at 32 s = 2 fps, at 128 s = 0.5 fps
    - low fps -> some quick actions can fall between snapshots, percision based on gap
- _thresholds_ (% of windows > 32s (< 2 fps) and > 64 s (< 1 fps))
- _plot_ -> duration histogram
- _bin lengths_ -> fixed cutoffs related to frame cap
    - short: <= 10s
    - med: 10-32s (full 2 fps)
    - long: > 32s (cap < 2 fps)

---
- *desicion* -> length_bin in gt_intervals
    - accuracy by length: do long events score worse? (fewer snapshots)
    - speedup by length: PTD's advantage grows with more boxes, then plateaus once boxes hit the 64 cap


In [0]:
FPS_MODEL, FRAME_CAP = 2, 64
PAD_S = 3.0 # sec of padding on each side

### interval duration

In [0]:
gt_intervals = gt_intervals.drop(columns=video_metadata.columns.drop("video_id"), errors="ignore")
gt_intervals = gt_intervals.merge(video_metadata, on="video_id", how="left")

gt_intervals["duration_s"] = (gt_intervals["end_frame"] - gt_intervals["start_frame"]) / gt_intervals["fps"]

d = gt_intervals["duration_s"]
q1, med, q3 = d.quantile([.25, .5, .75])
print(f"median {med:.1f} s | IQR {q3 - q1:.1f} s ({q1:.1f}–{q3:.1f}) | min {d.min():.1f} s | max {d.max():.1f} s")

display(gt_intervals.groupby("class")["duration_s"].agg(["median", "min", "max"]).round(1))

### window length

In [0]:
gt_intervals["window_s"] = gt_intervals["duration_s"] + 2 * PAD_S

### model frames

In [0]:
gt_intervals["model_frames"] = np.minimum(FRAME_CAP, np.floor(gt_intervals["window_s"] * FPS_MODEL)).astype(int)

### effective sampling

In [0]:
gt_intervals["eff_fps"]    = gt_intervals["model_frames"] / gt_intervals["window_s"]
gt_intervals["snap_gap_s"] = 1 / gt_intervals["eff_fps"]

display(gt_intervals.sort_values("window_s", ascending=False)
        [["video_id", "window_s", "model_frames", "eff_fps", "snap_gap_s"]].head(10).round(2))

`eff_fps` is 2.0 for any window up to 32 s, then falls: 64 / 128 s = 0.5

table shows 10 longest windows (most affected by snapshots)

### thresholds

In [0]:
for t in (32, 64):
    share = (gt_intervals["window_s"] > t).mean()
    n = (gt_intervals["window_s"] > t).sum()
    print(f"windows > {t} s (< {FRAME_CAP / t:.0f} fps): {n} intervals, {share:.1%}")

### plot: duration histogram

In [0]:
bins = np.logspace(np.log10(d.min()), np.log10(d.max()), 30)

plt.figure(figsize=(7, 4))
plt.hist(gt_intervals["duration_s"], bins=bins, edgecolor="navy", linewidth=0.1)
plt.xscale("log")

for cut, lbl in [(10, "10 s"), (32 - 2 * PAD_S, "2 fps drop"), (64 - 2 * PAD_S, "1 fps drop")]:
    plt.axvline(cut, linestyle="--", color="red")
    plt.text(cut, plt.ylim()[1] * 0.97, lbl,
             rotation=90, ha="right", va="top", fontsize=8, color="red")
plt.xlabel("event duration (s, log scale)"); plt.ylabel("# intervals")
plt.title("Event durations"); plt.tight_layout(); plt.show()

### bin lengths

In [0]:
gt_intervals["length_bin"] = pd.cut(gt_intervals["window_s"], bins=[0, 10, 32, np.inf], labels=["short", "med", "long"])

display(gt_intervals.groupby("length_bin", observed=True).agg(
    intervals=("video_id", "size"),
    median_window_s=("window_s", "median"),
    median_frames=("model_frames", "median"),
    median_eff_fps=("eff_fps", "median"),
).round(2))

### decision
save length_bin with gt_intervals so it can be used in later notebooks

In [0]:
out = gt_intervals.astype({"length_bin": str})
(spark.createDataFrame(out)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.default.gt_intervals"))

spark.createDataFrame(boxes).write.mode("overwrite").saveAsTable("workspace.default.gt_boxes")

## 3. annotation coverage
- _boxes per interval_ (median, IQR) -> how many GT boxes each event has
    - these are the points vIoU is scored on -> few boxes = noisy score (1 bad box shifts it a lot)
    - flag intervals with < 5 scored boxes as noisy
- _annotation stride_ (gap between consecutive boxes) -> mostly 16 frames (~0.53 s at 30 fps)
    - model samples every ~15 frames, boxes every 16 -> never line up exactly
    - match each snapshot to nearest box within 8 frames (half the stride) -> MATCH_TOLERANCE = 8
- _long gaps_ (% of gaps > 48 frames) -> target likely hidden or out of frame
    - official vIoU* counts frames where EITHER side has a box
    - model draws a box where GT has none -> that frame scores 0 -> tracking through occlusion is penalized
- _box span coverage_ ((last box - first box) / (official end - official start)) -> median 83%
    - official metric uses box span as the true interval, not the official start/end
    - e.g. Shoplifting010: boxes 813-861 vs official 750-920
---
- _decision_ -> has_long_gaps flag
    - state box-span definition in 5.4**

**vIoU (video intersection over union)***
main score for spatio temporal grounding, measures if model drew correct box at right time across whole clip.
- *IoU* (one frame) = overlap area / total area covered by either box -> 1 = perfect, 0 = no overlap
- *vIoU* = (sum of per-frame IoU over frames where **both** sides have a box) / (# frames where **either** side has a box)
    - top -> only frames that can be compared
    - bottom -> every frame anyone drew a box -> one-sided frames add to bottom with 0 on top
- model only scores well if it boxes right place (high IoU) at right time (few one side frames)

**why long gaps penalize the model**
- gap = target hidden (out of view), annotator drew no boxes
- model that keeps tracking through the gap draws boxes where GT has none -> each adds to bottom, nothing to top
- vIoU rewards models that stop boxing during obscursion, even though tracking through it seems better

In [0]:
NOISY_MIN = 5  # intervals w less than these boxes are noisy
MATCH_TOLERANCE = 8 # frames: half the 16 frame stride
LONG_GAP = 48  # frames: 3x the normal stride

### per interval box stats

In [0]:
b = boxes.sort_values(["video_id", "interval_id", "frame_number"]).copy()
b["gap"] = b.groupby(["video_id", "interval_id"])["frame_number"].diff()

per_int = b.groupby(["video_id", "interval_id"]).agg( # one row per interval- first, last box frame, largest gap, number gaps > 48
    box_first=("frame_number", "min"),
    box_last=("frame_number", "max"),
    max_gap=("gap", "max"),
    n_long_gaps=("gap", lambda g: (g > LONG_GAP).sum()), 
).reset_index()

gt_intervals = gt_intervals.drop(columns=per_int.columns.drop(["video_id", "interval_id"]), errors="ignore")
gt_intervals = gt_intervals.merge(per_int, on=["video_id", "interval_id"], how="left")

### boxes per interval

In [0]:
nb = gt_intervals["n_boxes"]
q1, med, q3 = nb.quantile([.25, .5, .75])
print(f"boxes per interval: median {med:.0f} | IQR {q3 - q1:.0f} ({q1:.0f}–{q3:.0f}) | min {nb.min()} | max {nb.max()}")

gt_intervals["is_noisy"] = gt_intervals["n_boxes"] < NOISY_MIN
print(f"noisy (< {NOISY_MIN} boxes): {gt_intervals['is_noisy'].sum()} intervals")
display(gt_intervals.loc[gt_intervals["is_noisy"], ["video_id", "interval_id", "class", "n_boxes"]])

### annotation stride

In [0]:
gaps = b["gap"].dropna() # NaN at first box gets removed from initial groupby
stride = gaps.mode()[0] # most freq gap
print(f"most common stride: {stride:.0f} frames (~{stride / 30:.2f} s at 30 fps)")
print(gaps.value_counts().head(8))

### snapshot to box matching

In [0]:
def nearest_box_dist(row):
    box_f = b.loc[(b.video_id == row.video_id) & (b.interval_id == row.interval_id), "frame_number"].to_numpy() # one interval box frame #s, in order
    snaps = np.linspace(row.start_frame - PAD_S * row.fps, row.end_frame + PAD_S * row.fps, row.model_frames) # evenly spaces snapshot positions across window
    snaps = snaps[(snaps >= box_f[0]) & (snaps <= box_f[-1])] # keep inside box span
    idx = np.clip(np.searchsorted(box_f, snaps), 1, len(box_f) - 1) # find where each snapshot slots into sorted box frames (box on either side = nearest neighbour)
    return np.minimum(np.abs(snaps - box_f[idx - 1]), np.abs(snaps - box_f[idx])) # keep pos inside array, gets closest of two neighbours

dists = np.concatenate([nearest_box_dist(r) for r in gt_intervals.itertuples() if r.n_boxes > 1]) # loops over rows, if r.n_boxes > 1, skips interval with one box that have no gaps to match
print(f"snapshots within {MATCH_TOLERANCE} frames of a box: {(dists <= MATCH_TOLERANCE).mean():.1%}")
print(f"median distance: {np.median(dists):.1f} frames")

`MATCH_TOLERANCE = 8` works since most snapshots fall within 8 frames

### long gaps

In [0]:
print(f"gaps > {LONG_GAP} frames: {(gaps > LONG_GAP).mean():.1%} of all gaps") # share of all gaps that are long

gt_intervals["has_long_gaps"] = gt_intervals["n_long_gaps"] > 0 # decision flag -> if true interval has at least one gap over 48 frames (where vIoU penalizes target)

print(f"intervals with a long gap: {gt_intervals['has_long_gaps'].sum()} of {len(gt_intervals)}")

display(gt_intervals.loc[gt_intervals["has_long_gaps"], ["video_id", "interval_id", "class", "max_gap", "n_long_gaps"]].sort_values("max_gap", ascending=False))

### box span coverage

In [0]:
official_len = gt_intervals["end_frame"] - gt_intervals["start_frame"]
gt_intervals["span_coverage"] = (gt_intervals["box_last"] - gt_intervals["box_first"]) / official_len

print(f"box span coverage: median {gt_intervals['span_coverage'].median():.0%}")
display(gt_intervals.loc[gt_intervals.video_id == "Shoplifting010_x264",
        ["interval_id", "start_frame", "end_frame", "box_first", "box_last", "span_coverage"]]) # random video example
gt_intervals = gt_intervals.astype({"start_frame": int, "end_frame": int})

In [0]:
plt.hist(gt_intervals["span_coverage"], bins=20, edgecolor="navy", linewidth=0.8)
plt.xlabel("box span / official interval"); plt.ylabel("# intervals")
plt.title("Box span coverage"); plt.show()

- in a single event:
    - official interval: when crime happen
    - box span: when anomaly/person is visible and in box
- coverage = how much official interval boxes fill (1= all)
- some events have box fill full event, while some have person only partially visible
- *decision*: flag low coverage (< 0.5), to later check if low score come bc of this.



## 4. spatial properties (box)
- *box area* (box w × h / 320 × 240) -> how much of the frame each target covers
    - small boxes -> IoU harsher, PTD upscales frames (no extra detail)
- *size bins* -> small < 5%, medium 5–25%, large > 25%
- *aspect ratio* (w / h) -> tall = person, wide = vehicle / scene
    - check each class's query fits its target
- *box centre heatmap* -> where boxes sit in the frame (CCTV aims at the centre)
- *edge contact* (% of boxes touching the border) -> target partly outside the frame

---

- *decision* -> size_bin + edge_share in gt_intervals
    - accuracy by size: do small targets score worse?
    - edge vs non-edge: does edge contact lower IoU?
    - PTD's real gain = margin above the central-box baseline


In [0]:
EDGE_TOL = 2  # px from the border that counts as "touching"
SIZE_CUTS = [0, 0.05, 0.25, 1.0] # small < 5%, medium 5–25%, large > 25%
FRAME_W, FRAME_H = 320, 240 # frame size of all videos in dataset

### box area

In [0]:
boxes["w"] = boxes["x2"] - boxes["x1"]
boxes["h"] = boxes["y2"] - boxes["y1"]
boxes["area_frac"] = (boxes["w"] * boxes["h"]) / (FRAME_W * FRAME_H)

a = boxes["area_frac"]
print(f"box area / frame: median {a.median():.0%} | p10 {a.quantile(.1):.0%} | p90 {a.quantile(.9):.0%}")

### size bins

In [0]:
boxes["size_bin"] = pd.cut(boxes["area_frac"], 
                           bins=SIZE_CUTS,
                           labels=["small", "medium", "large"], include_lowest=True) # put each box into group, when 0 lands in small
print(boxes["size_bin"].value_counts(normalize=True).round(2)) # shares

### aspect ratio

In [0]:
boxes["aspect"] = boxes["w"] / boxes["h"]
display(boxes.groupby("class").agg(
    median_aspect=("aspect", "median"),
    median_area=("area_frac", "median"),
    ).round(2).sort_values("median_aspect")) # aspect < 1 is tall box (person), > 1 is wide box (scene object ex. car)
    

### heatmap: box centre

#### box-level

In [0]:
# centre points
cx = (boxes["x1"] + boxes["x2"]) / 2
cy = (boxes["y1"] + boxes["y2"]) / 2

plt.figure(figsize=(6, 4.5))
plt.hist2d(cx, cy, bins=[16, 12], range=[[0, FRAME_W], [0, FRAME_H]], cmap="Blues")
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.colorbar(label="# boxes")
plt.xlabel("x (px)"); plt.ylabel("y (px)"); plt.title("Box centres")
plt.tight_layout(); plt.show()

- boxes cluster in centre of frame, edges are emptier
- fixed central box scores well because thats where box mostly is
    - PTD needs to beat by finding actual target
-  darkest cell partly inflated by long intervals (many boxes) -> see interval-level

#### interval-level

In [0]:
c = boxes.assign(cx=cx, cy=cy).groupby(["video_id", "interval_id"])[["cx", "cy"]].mean()

plt.figure(figsize=(6, 4.5))
plt.hist2d(c["cx"], c["cy"], bins=[16, 12], range=[[0, FRAME_W], [0, FRAME_H]], cmap="Blues")
plt.gca().invert_yaxis(); plt.gca().set_aspect("equal")
plt.colorbar(label="# intervals")
plt.xlabel("x (px)"); plt.ylabel("y (px)"); plt.title("Box centres (one per interval)")
plt.tight_layout(); plt.show()

- one point per interval -> central cluster stays, so it's the dataset, not a few long videos
- event centres spread wide (x ~25–280) but not tall (y ~50–180) -> cameras mounted high, angled down
- busiest cell ~10 of 156 intervals -> events spread across the middle, not piled on one spot
- -> a fixed central box overlaps many targets by chance -> report PTD's margin above it

### edge contact

In [0]:
boxes["touches_edge"] = ((boxes["x1"] <= EDGE_TOL) | (boxes["y1"] <= EDGE_TOL) |
                         (boxes["x2"] >= FRAME_W - EDGE_TOL) | (boxes["y2"] >= FRAME_H - EDGE_TOL)) # checks one side of box against border, any side counts

print(f"boxes touching the border: {boxes['touches_edge'].mean():.1%}") # share it is true 
display(boxes.groupby("class")["touches_edge"].mean().sort_values(ascending=False).round(2))

### decision
add to gt_intervals

In [0]:
# summarize each interval boxes: typical size (median_area_frac, then binned into size_bin) and the share touching the edge (edge_share)

sp = boxes.groupby(["video_id", "interval_id"]).agg(
    median_area_frac=("area_frac", "median"),
    edge_share=("touches_edge", "mean"),
    ).reset_index()

sp["size_bin"] = pd.cut(sp["median_area_frac"], bins=SIZE_CUTS,
                        labels=["small", "medium", "large"], include_lowest=True)

gt_intervals = gt_intervals.drop(columns=sp.columns.drop(["video_id", "interval_id"]), errors="ignore")
gt_intervals = gt_intervals.merge(sp, on=["video_id", "interval_id"], how="left") 

In [0]:
out = gt_intervals.astype({"length_bin": str, "size_bin": str})
(spark.createDataFrame(out)
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.default.gt_intervals"))

(spark.createDataFrame(boxes.astype({"size_bin": str}))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("workspace.default.gt_boxes"))

## 5. Motion within each tube